# 📎 Complementar da Aula 3: filtros extras e Git na prática

O gestor volta com mais três pedidos:

1. *"Me traga os clientes entre 25 e 40 anos, sem escrever duas comparações."*
2. *"Quero a lista de colaboradores cujo cargo tem a palavra 'Analista'."*
3. *"Preciso testar uma mudança na planilha sem estragar a versão oficial."*

Este notebook **não é apresentado ao vivo**. Ele guarda o que ficou de fora da aula: filtros que encurtam o código e o Git na prática (`.gitignore`, branches, merge e conflito, de verdade). Faça no seu ritmo, e as respostas dos exercícios estão escondidas.

## 🗺️ Roteiro

- 🎯 Filtros extras: `between`, `query`, `str.contains`, `loc` com máscara, `nlargest`
- 🌿 Git na prática: `.gitignore`, branch, merge e conflito
- 🧾 Resumo
- 🏋️ Exercícios extras por área (💰 💼 👥) e de Git

**No fim você vai saber:**
1. escrever filtros mais curtos e legíveis;
2. procurar texto dentro de uma coluna;
3. impedir que arquivos sensíveis entrem no histórico do Git;
4. trabalhar numa branch e juntar de volta à principal;
5. resolver um conflito de merge.

## ⚙️ Configuração

Vou carregar a mesma base de cartão de crédito da aula e a tabela de colaboradores.

In [1]:
# Esta célula instala o kagglehub, caso ainda não esteja disponível.
!pip install -q kagglehub


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Esta célula importa as bibliotecas e mostra as versões.
import os
import shutil
import stat
from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

print("pandas:", pd.__version__)

pandas: 3.0.6


In [3]:
# Esta célula baixa a base de cartão de crédito, renomeia as colunas e cria o uso do limite.
caminho = kagglehub.dataset_download("uciml/default-of-credit-card-clients-dataset")
clientes = pd.read_csv(Path(caminho) / "UCI_Credit_Card.csv")
clientes = clientes.rename(columns={
    "ID": "id_cliente", "LIMIT_BAL": "limite_credito", "SEX": "sexo",
    "EDUCATION": "escolaridade", "MARRIAGE": "estado_civil", "AGE": "idade",
    "PAY_0": "atraso_set", "PAY_2": "atraso_ago", "PAY_3": "atraso_jul",
    "PAY_4": "atraso_jun", "PAY_5": "atraso_mai", "PAY_6": "atraso_abr",
    "BILL_AMT1": "fatura_set", "BILL_AMT2": "fatura_ago", "BILL_AMT3": "fatura_jul",
    "BILL_AMT4": "fatura_jun", "BILL_AMT5": "fatura_mai", "BILL_AMT6": "fatura_abr",
    "PAY_AMT1": "pago_set", "PAY_AMT2": "pago_ago", "PAY_AMT3": "pago_jul",
    "PAY_AMT4": "pago_jun", "PAY_AMT5": "pago_mai", "PAY_AMT6": "pago_abr",
    "default.payment.next.month": "inadimplente",
})
clientes["uso_limite"] = clientes["fatura_set"] / clientes["limite_credito"]
clientes.shape

(30000, 26)

In [4]:
# Esta célula cria a tabela de colaboradores, com nome, cargo, setor e salário.
folha = pd.DataFrame({
    "colaborador": ["Marta", "Paulo", "Renata", "Sérgio", "Tânia", "Vera"],
    "cargo": ["Analista Financeiro", "Gerente Comercial", "Analista de RH",
              "Assistente Comercial", "Coordenadora Financeira", "Analista Comercial"],
    "setor": ["Financeiro", "Comercial", "RH", "Comercial", "Financeiro", "Comercial"],
    "salario": [6500.0, 12000.0, 6800.0, 3500.0, 9000.0, 6200.0],
}).set_index("colaborador")
folha

,cargo,setor,salario
colaborador,,,
Marta,Analista Financeiro,Financeiro,6500.0
Paulo,Gerente Comercial,Comercial,12000.0
Renata,Analista de RH,RH,6800.0
Sérgio,Assistente Comercial,Comercial,3500.0
Tânia,Coordenadora Financeira,Financeiro,9000.0
Vera,Analista Comercial,Comercial,6200.0


## 🎯 Filtros que ficaram de fora

### `between`: um intervalo sem repetir a coluna

Na aula escrevemos `(idade >= 25) & (idade <= 40)`. O `between` faz isso numa chamada só, **incluindo as duas pontas**.

In [5]:
# Esta célula conta os clientes entre 25 e 40 anos, de duas maneiras.
jeito_longo = ((clientes["idade"] >= 25) & (clientes["idade"] <= 40)).sum()
jeito_curto = clientes["idade"].between(25, 40).sum()
print(jeito_longo, jeito_curto)

19041 19041


Os dois jeitos dão o mesmo número. O `between` é mais curto e mais difícil de errar.

### `query`: escrever o filtro como uma frase

O `query` recebe o filtro **como texto**. Dentro dele, `and`, `or` e `not` funcionam normalmente, e não precisa repetir o nome da tabela.

In [6]:
# Esta célula filtra clientes com mais de 60 anos e limite acima de 200 mil usando query.
grupo = clientes.query("idade > 60 and limite_credito > 200000")
len(grupo)

110

Dá o mesmo resultado do `&` da aula. Para usar uma **variável** do Python dentro do `query`, coloque `@` na frente do nome.

In [7]:
# Esta célula usa uma variável do Python dentro do query, com @.
idade_minima = 60
limite_minimo = 200000
clientes.query("idade > @idade_minima and limite_credito > @limite_minimo").shape[0]

110

### `str.contains`: procurar texto dentro de uma coluna

Para colunas de texto, o prefixo `.str` libera operações como `contains` (contém), `startswith` (começa com) e `endswith` (termina com).

In [8]:
# Esta célula filtra os colaboradores cujo cargo contém a palavra Analista.
folha[folha["cargo"].str.contains("Analista")]

,cargo,setor,salario
colaborador,,,
Marta,Analista Financeiro,Financeiro,6500.0
Renata,Analista de RH,RH,6800.0
Vera,Analista Comercial,Comercial,6200.0


Vieram **Marta, Renata e Vera**. Atenção: o `contains` diferencia maiúsculas de minúsculas. Para ignorar a diferença, use `case=False`.

In [9]:
# Esta célula procura "financeir" sem se importar com maiúsculas e minúsculas.
folha[folha["cargo"].str.contains("financeir", case=False)]

,cargo,setor,salario
colaborador,,,
Marta,Analista Financeiro,Financeiro,6500.0
Tânia,Coordenadora Financeira,Financeiro,9000.0


Vieram **Marta** (Analista Financeiro) e **Tânia** (Coordenadora Financeira). Procurar só "financeir" pegou as duas formas, masculina e feminina.

### `loc` com máscara: filtrar e escolher colunas de uma vez

Dentro do `loc`, a primeira parte pode ser uma máscara e a segunda, a lista de colunas.

In [10]:
# Esta célula pega só o id e o limite dos clientes com mais de 60 anos, com loc e uma máscara.
clientes.loc[clientes["idade"] > 60, ["id_cliente", "limite_credito"]].head()

,id_cliente,limite_credito
102,103,480000.0
246,247,250000.0
366,367,50000.0
387,388,80000.0
1029,1030,30000.0


Essa forma também é a **maneira segura de alterar valores**: `df.loc[máscara, "coluna"] = valor`. Vou dar um bônus de 10% aos colaboradores do Comercial.

In [11]:
# Esta célula cria a coluna bonus e preenche 10% do salário só para o setor Comercial.
folha["bonus"] = 0.0
folha.loc[folha["setor"] == "Comercial", "bonus"] = folha["salario"] * 0.10
folha

,cargo,setor,salario,bonus
colaborador,,,,
Marta,Analista Financeiro,Financeiro,6500.0,0.0
Paulo,Gerente Comercial,Comercial,12000.0,1200.0
Renata,Analista de RH,RH,6800.0,0.0
Sérgio,Assistente Comercial,Comercial,3500.0,350.0
Tânia,Coordenadora Financeira,Financeiro,9000.0,0.0
Vera,Analista Comercial,Comercial,6200.0,620.0


Só Paulo (1.200), Sérgio (350) e Vera (620) receberam bônus, e os demais ficaram em zero.

### `nlargest` e `nsmallest`: os maiores e os menores

Em vez de ordenar e pegar o começo, o `nlargest` já devolve os N maiores.

In [12]:
# Esta célula mostra os três clientes com maior uso do limite.
clientes.nlargest(3, "uso_limite")[["id_cliente", "limite_credito", "fatura_set", "uso_limite"]]

,id_cliente,limite_credito,fatura_set,uso_limite
3796,3797,20000.0,129106.0,6.4553
7508,7509,10000.0,53095.0,5.3095
20635,20636,20000.0,82812.0,4.1406


O cliente **3797** aparece de novo no topo, com uso de 6,46 vezes o limite, como vimos na aula.

## 🌿 Git na prática

Na aula eu mostrei o ciclo `status → add → commit → log`. Aqui você roda tudo sozinho, num repositório de treino. Primeiro, uma função de apoio para limpar pastas de treino (o Git deixa arquivos protegidos que atrapalham a exclusão).

### 🔧 Função pronta

Esta função apaga uma pasta de treino se ela já existir e cria uma pasta nova, para você poder repetir o notebook quantas vezes quiser.

In [13]:
# Função pronta: recria uma pasta de treino vazia, apagando a anterior se existir.
def recriar_pasta(nome):
    def liberar(funcao, caminho, _):
        os.chmod(caminho, stat.S_IWRITE)
        funcao(caminho)

    pasta = Path(nome)
    if pasta.exists():
        shutil.rmtree(pasta, onerror=liberar)
    pasta.mkdir(parents=True)
    return pasta

### 1. `.gitignore`: o que o Git deve ignorar

Há arquivos que **nunca** devem entrar no histórico: senhas, chaves de acesso, bases de clientes. O arquivo `.gitignore` lista o que o Git deve fingir que não vê.

Vou criar um repositório com três arquivos: um código, uma base de clientes e um arquivo de senhas.

In [14]:
# Esta célula cria um repositório de treino com um script, uma base de clientes e um arquivo de senhas.
pasta = recriar_pasta("treino_gitignore")
(pasta / "analise.py").write_text("print('olá')\n", encoding="utf-8")
(pasta / "clientes.csv").write_text("id,nome\n1,Ana\n", encoding="utf-8")
(pasta / "senhas.txt").write_text("usuario=admin\nsenha=1234\n", encoding="utf-8")

!git -C treino_gitignore init -b main
!git -C treino_gitignore config user.name "Analista Exemplo"
!git -C treino_gitignore config user.email "analista@exemplo.com"
!git -C treino_gitignore status --short

Initialized empty Git repository in H:/Ada- Tech/Tecnicas de Programacao/aula-03-pandas-fundamentos/treino_gitignore/.git/


?? analise.py
?? clientes.csv
?? senhas.txt


O `??` na frente de cada arquivo quer dizer `untracked`: o Git viu os **três**. Se eu fizer `git add .` agora, a senha e a base de clientes vão para o histórico. Vou criar o `.gitignore` antes.

In [15]:
# Esta célula cria o .gitignore listando a base de clientes e o arquivo de senhas.
(pasta / ".gitignore").write_text("senhas.txt\nclientes.csv\n", encoding="utf-8")
!git -C treino_gitignore status --short

?? .gitignore
?? analise.py


Agora o Git só enxerga o `analise.py` e o próprio `.gitignore`. Os outros dois sumiram da lista.

💡 Dá para usar curingas: `*.csv` ignora **todos** os CSV, e `dados/` ignora a pasta inteira. **Atenção:** o `.gitignore` só vale para arquivos que **ainda não entraram** no histórico. Se a senha já foi commitada, ela continua lá.

### 2. Branch e merge

Uma **branch** é uma linha paralela de trabalho. Vou criar um repositório com um commit na `main`, abrir uma branch para testar uma coluna nova e trazer o trabalho de volta.

In [16]:
# Esta célula cria um repositório de treino com um primeiro commit na branch main.
pasta = recriar_pasta("treino_branch")
(pasta / "vendas.csv").write_text("agencia,vendas\nCentro,100\nNorte,80\n", encoding="utf-8")

!git -C treino_branch init -b main
!git -C treino_branch config user.name "Analista Exemplo"
!git -C treino_branch config user.email "analista@exemplo.com"
!git -C treino_branch add vendas.csv
!git -C treino_branch commit -m "Adiciona vendas por agência"


Initialized empty Git repository in H:/Ada- Tech/Tecnicas de Programacao/aula-03-pandas-fundamentos/treino_branch/.git/


[main (root-commit) 4408735] Adiciona vendas por agência
 1 file changed, 3 insertions(+)
 create mode 100644 vendas.csv


In [17]:
# Esta célula cria a branch teste-meta e muda para ela.
!git -C treino_branch switch -c teste-meta
!git -C treino_branch branch

Switched to a new branch 'teste-meta'


  main
* teste-meta


O `*` marca a branch em que estou: `teste-meta`. Vou alterar o arquivo **só nela** e fazer um commit.

In [18]:
# Esta célula acrescenta uma coluna de meta ao arquivo e guarda o commit na branch teste-meta.
(pasta / "vendas.csv").write_text("agencia,vendas,meta\nCentro,100,120\nNorte,80,90\n", encoding="utf-8")
!git -C treino_branch add vendas.csv
!git -C treino_branch commit -m "Inclui coluna de meta"


[teste-meta 7f0efe5] Inclui coluna de meta
 1 file changed, 3 insertions(+), 3 deletions(-)


Agora volto para a `main`. O arquivo **volta ao que era**: a coluna de meta só existe na outra branch.

In [19]:
# Esta célula volta para a main e mostra o conteúdo do arquivo nela.
!git -C treino_branch switch main
print((pasta / "vendas.csv").read_text(encoding="utf-8"))

agencia,vendas
Centro,100
Norte,80



Switched to branch 'main'


O arquivo na `main` ainda não tem a coluna `meta`. Testei sem mexer na versão oficial. Gostei do resultado, então faço o **merge**.

In [20]:
# Esta célula traz o trabalho da branch teste-meta para a main e mostra o histórico em forma de gráfico.
!git -C treino_branch merge teste-meta
!git -C treino_branch log --oneline --graph --all

Updating 4408735..7f0efe5
Fast-forward
 vendas.csv | 6 +++---
 1 file changed, 3 insertions(+), 3 deletions(-)


* 7f0efe5 Inclui coluna de meta
* 4408735 Adiciona vendas por agência


A `main` agora tem a coluna de meta, e o histórico mostra os dois commits. Como a `main` não tinha mudado nesse meio-tempo, o Git só precisou "avançar" a linha (*fast-forward*), sem conflito.

### 3. Conflito de merge, de verdade

Agora o caso difícil: **duas branches mudam a mesma linha**. Vou criar o cenário: a `main` altera a meta do Norte para 95, e a outra branch altera a mesma linha para 100.

In [21]:
# Esta célula prepara o cenário: um commit inicial e duas branches que mudam a mesma linha.
pasta = recriar_pasta("treino_conflito")
!git -C treino_conflito init -b main
!git -C treino_conflito config user.name "Analista Exemplo"
!git -C treino_conflito config user.email "analista@exemplo.com"

(pasta / "meta.csv").write_text("agencia,meta\nNorte,90\n", encoding="utf-8")
!git -C treino_conflito add meta.csv
!git -C treino_conflito commit -m "Meta inicial"

!git -C treino_conflito switch -c ajuste-bruno
(pasta / "meta.csv").write_text("agencia,meta\nNorte,100\n", encoding="utf-8")
!git -C treino_conflito commit -am "Bruno muda meta para 100"

!git -C treino_conflito switch main
(pasta / "meta.csv").write_text("agencia,meta\nNorte,95\n", encoding="utf-8")
!git -C treino_conflito commit -am "Ana muda meta para 95"


Initialized empty Git repository in H:/Ada- Tech/Tecnicas de Programacao/aula-03-pandas-fundamentos/treino_conflito/.git/


[main (root-commit) 48b49ef] Meta inicial
 1 file changed, 2 insertions(+)
 create mode 100644 meta.csv


Switched to a new branch 'ajuste-bruno'


[ajuste-bruno 8e37988] Bruno muda meta para 100
 1 file changed, 1 insertion(+), 1 deletion(-)


Switched to branch 'main'


[main fdd96cd] Ana muda meta para 95
 1 file changed, 1 insertion(+), 1 deletion(-)


A Ana mudou a meta para 95 na `main`, e o Bruno para 100 na `ajuste-bruno`. Vamos tentar juntar. **Vou provocar o conflito de propósito:**

In [22]:
# Esta célula tenta juntar a branch do Bruno na main, o que gera um conflito.
!git -C treino_conflito merge ajuste-bruno

Auto-merging meta.csv
CONFLICT (content): Merge conflict in meta.csv
Automatic merge failed; fix conflicts and then commit the result.


**Como ler:** `CONFLICT (content): Merge conflict in meta.csv` e `Automatic merge failed; fix conflicts and then commit the result`. Traduzindo: "mexeram na mesma linha e eu não sei qual vale; resolva e depois faça o commit". Vamos olhar o arquivo.

In [23]:
# Esta célula mostra o arquivo com as marcas de conflito.
print((pasta / "meta.csv").read_text(encoding="utf-8"))

agencia,meta
<<<<<<< HEAD
Norte,95
Norte,100
>>>>>>> ajuste-bruno



Entre `<<<<<<< HEAD` e `=======` está a versão da `main` (95), e entre `=======` e `>>>>>>> ajuste-bruno` a do Bruno (100). Numa situação real, vocês conversariam para decidir. Vamos supor que a meta correta é **100**. Eu apago as marcas, deixo o valor certo e faço o commit.

In [24]:
# Esta célula resolve o conflito escrevendo o valor final e conclui o merge com um commit.
(pasta / "meta.csv").write_text("agencia,meta\nNorte,100\n", encoding="utf-8")
!git -C treino_conflito add meta.csv
!git -C treino_conflito commit -m "Resolve conflito: meta final é 100"
!git -C treino_conflito log --oneline --graph --all

[main b38d1ff] Resolve conflito: meta final é 100


*   b38d1ff Resolve conflito: meta final é 100
|\  
| * 8e37988 Bruno muda meta para 100
* | fdd96cd Ana muda meta para 95
|/  
* 48b49ef Meta inicial


O histórico mostra as duas linhas se encontrando no commit final. **Conflito resolvido.** Em resumo: abrir o arquivo, escolher o valor, tirar as marcas, `add` e `commit`.

### 4. Usando o GitHub no seu computador (passo a passo)

Para fazer o `push` de verdade, faça no terminal do seu computador (o do VS Code serve):

1. **Instale o Git** (git-scm.com) e configure seu nome uma vez só:
   ```bash
   git config --global user.name "Seu Nome"
   git config --global user.email "seu-email-pessoal@exemplo.com"
   ```
2. No site do GitHub, clique em **New** para criar um repositório. Dê um nome (por exemplo, `meu-primeiro-repo`) e marque **Add a README**.
3. Clique em **Code** e copie o endereço HTTPS. No terminal, baixe o repositório:
   ```bash
   git clone https://github.com/SEU_USUARIO/meu-primeiro-repo.git
   cd meu-primeiro-repo
   ```
4. Crie ou edite um arquivo, e depois:
   ```bash
   git add .
   git commit -m "Minha primeira mudança"
   git push
   ```
5. Na primeira vez, o Git abre uma janela do navegador para você **entrar no GitHub**. Depois disso, é automático.
6. Atualize a página do repositório e veja o commit lá. 🎉

💡 O VS Code também tem um painel **Source Control** (ícone de ramificação, na barra lateral) que faz `add`, `commit` e `push` com cliques, usando os mesmos comandos por baixo.

⚠️ Lembre: **e-mail pessoal** e **nada de dados da empresa** em repositório público.

## 🧾 Resumo

| Comando | Para que serve |
|---|---|
| `serie.between(a, b)` | Valores entre `a` e `b`, incluindo as pontas |
| `df.query("idade > 60 and limite > 200000")` | Filtro escrito como texto; `@variavel` usa variáveis do Python |
| `serie.str.contains("texto", case=False)` | Linhas cujo texto contém o trecho |
| `df.loc[máscara, ["a", "b"]]` | Filtrar linhas e escolher colunas de uma vez |
| `df.loc[máscara, "col"] = valor` | Alterar valores de forma segura |
| `df.nlargest(3, "col")` / `nsmallest` | Os maiores / os menores |
| `.gitignore` | Lista do que o Git deve ignorar |
| `git switch -c nome` | Cria uma branch e vai para ela |
| `git switch nome` | Muda de branch |
| `git merge nome` | Traz o trabalho da branch para a atual |
| `git log --oneline --graph --all` | Histórico em forma de gráfico |

### Mensagens que aparecem

| Mensagem | O que significa |
|---|---|
| `CONFLICT (content): Merge conflict in arquivo` | Duas branches mudaram a mesma linha; é preciso escolher o valor |
| `Automatic merge failed; fix conflicts and then commit the result` | Resolva o arquivo, faça `add` e `commit` |
| `Already up to date.` | Não há nada novo para juntar |

## 🏋️ Exercícios extras por área

Escreva a resposta na célula vazia e compare com a recolhida.

### 💰 Financeiro

**Exercício 1.** Quantos clientes têm **entre 30 e 40 anos** e **fatura maior que o limite** (`uso_limite` acima de 1)? Use `between`.

In [25]:
# sua resposta aqui


In [26]:
# Resposta
((clientes["idade"].between(30, 40)) & (clientes["uso_limite"] > 1)).sum()


np.int64(729)

**Exercício 2.** Reescreva o exercício anterior usando `query`.

In [27]:
# sua resposta aqui


In [28]:
# Resposta
len(clientes.query("idade >= 30 and idade <= 40 and uso_limite > 1"))


729

### 💼 Comercial

**Exercício 3.** Mostre os 5 clientes com o **maior limite de crédito**, com `id_cliente`, `limite_credito` e `idade`.

In [29]:
# sua resposta aqui


In [30]:
# Resposta
clientes.nlargest(5, "limite_credito")[["id_cliente", "limite_credito", "idade"]]


,id_cliente,limite_credito,idade
2197,2198,1000000.0,47
14919,14920,800000.0,53
18006,18007,800000.0,46
24320,24321,780000.0,48
28715,28716,780000.0,41


**Exercício 4.** Crie a coluna `perfil_campanha` com `True` para quem tem **menos de 30 anos** e **uso do limite acima de 50%**. Depois conte quantos clientes têm `True`.

*Dica: uma comparação entre colunas já devolve `True` e `False`, é só guardar numa coluna nova.*

In [31]:
# sua resposta aqui


In [32]:
# Resposta
clientes["perfil_campanha"] = (clientes["idade"] < 30) & (clientes["uso_limite"] > 0.5)
clientes["perfil_campanha"].sum()


np.int64(4487)

### 👥 RH

**Exercício 5.** Quais colaboradores têm cargo que **começa com** "Assistente" ou "Gerente"?

*Dica: `str.startswith` aceita uma tupla de opções, como `("Assistente", "Gerente")`.*

In [33]:
# sua resposta aqui


In [34]:
# Resposta
folha[folha["cargo"].str.startswith(("Assistente", "Gerente"))]


,cargo,setor,salario,bonus
colaborador,,,,
Paulo,Gerente Comercial,Comercial,12000.0,1200.0
Sérgio,Assistente Comercial,Comercial,3500.0,350.0


**Exercício 6.** Dê um aumento de 8% só para quem ganha **menos de R$ 7.000**, numa coluna nova `salario_novo` (quem ganha mais fica igual). Mostre a tabela.

*Dica: comece copiando o salário para a coluna nova, e depois use `loc` com a máscara.*

In [35]:
# sua resposta aqui


In [36]:
# Resposta
folha["salario_novo"] = folha["salario"]
folha.loc[folha["salario"] < 7000, "salario_novo"] = folha["salario"] * 1.08
folha[["salario", "salario_novo"]]


,salario,salario_novo
colaborador,,
Marta,6500.0,7020.0
Paulo,12000.0,12000.0
Renata,6800.0,7344.0
Sérgio,3500.0,3780.0
Tânia,9000.0,9000.0
Vera,6200.0,6696.0


### 🌿 Git

**Exercício 7.** Crie o repositório `treino_casa`, faça um commit com o arquivo `lista.txt` (com a linha `item 1`), crie a branch `novo-item`, acrescente a linha `item 2` e faça o commit nela. Volte para a `main` e confira, lendo o arquivo, que ela ainda só tem o item 1.

In [37]:
# sua resposta aqui


In [38]:
# Resposta
pasta = recriar_pasta("treino_casa")
!git -C treino_casa init -b main
!git -C treino_casa config user.name "Analista Exemplo"
!git -C treino_casa config user.email "analista@exemplo.com"

(pasta / "lista.txt").write_text("item 1\n", encoding="utf-8")
!git -C treino_casa add lista.txt
!git -C treino_casa commit -m "Cria lista com item 1"

!git -C treino_casa switch -c novo-item
(pasta / "lista.txt").write_text("item 1\nitem 2\n", encoding="utf-8")
!git -C treino_casa commit -am "Acrescenta item 2"

!git -C treino_casa switch main
print((pasta / "lista.txt").read_text(encoding="utf-8"))


Initialized empty Git repository in H:/Ada- Tech/Tecnicas de Programacao/aula-03-pandas-fundamentos/treino_casa/.git/


[main (root-commit) 733a485] Cria lista com item 1
 1 file changed, 1 insertion(+)
 create mode 100644 lista.txt


Switched to a new branch 'novo-item'


[novo-item 0af3958] Acrescenta item 2
 1 file changed, 1 insertion(+)


item 1



Switched to branch 'main'


## 🚀 Próxima aula

Na aula 4 vamos **limpar** os dados (valores faltantes, duplicados, categorias bagunçadas) e **resumir por grupo** com `groupby` e `pivot_table`. Aproveite este notebook para praticar. Quanto mais você repetir, mais natural fica.